In [1]:
!pip install ultralytics roboflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 43.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.9/207.9 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 88.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 107.8 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.92
    Uninstalling opencv-python-headless-4.13.0.92:
      Successfully uninstalled opencv-python-headless-4.13.0.92
  Attempting uninstall: idna
    Found existing installation: idna 3.15
    Uninstalling idna-3.15:
      Successfully uninstalled idna-3.15


In [2]:
import os
import numpy as np
import cv2
from roboflow import Roboflow
from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [3]:
rf = Roboflow(api_key="4ogzuUvRWwTKTqAXgq25")
project = rf.workspace("mohamed-90rl9").project("tail-clamp-detection")
version = project.version(2)
dataset = version.download("yolov11")


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Tail-clamp-detection-2 in yolov11:: 100%|██████████| 115/115 [00:00<00:00, 4189.13it/s]


In [4]:
# Load YOLOv11m
model = YOLO("yolo11m.pt")

# Train
model.train(
    data="/content/Tail-clamp-detection-2/data.yaml",
    epochs=200,
    imgsz=640,
    project="Tail-clamp-detection-2"
)

Ultralytics 8.4.56 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Tail-clamp-detection-2/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=200, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, 

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x79c8e80aa240>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.04804

In [5]:
model = YOLO(
    "/content/runs/detect/Tail-clamp-detection-2/train/weights/best.pt"
    )

In [6]:
def save_video_detection(video_path, video_name):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise FileNotFoundError(f"Cannot open video: {video_path}")

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = int(cap.get(cv2.CAP_PROP_FPS)) or 30

    out = cv2.VideoWriter(f"{video_name}.mp4", cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))

    def get_center(box):
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        return ((x1 + x2) / 2, (y1 + y2) / 2)

    def euclidean(p1, p2):
        return np.sqrt((p1[0] - p2[0])**2 + (p1[1] - p2[1])**2)

    def tail_diagonal(box):
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        return np.sqrt((x2 - x1)**2 + (y2 - y1)**2)

    try:
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break

            results = model(frame, conf=0.5)
            result = results[0]

            clamps = [box for box in result.boxes if int(box.cls) == 0]
            tails  = [box for box in result.boxes if int(box.cls) == 1]

            clamp_count = len(clamps)
            tail_count  = len(tails)

            annotated_frame = result.plot()

            shortest_tail = None
            longest_tail  = None

            # Distance between shortest and longest tail
            dist_tails_text = "N/A"
            if len(tails) >= 2:
                sorted_tails  = sorted(tails, key=tail_diagonal)
                shortest_tail = sorted_tails[0]
                longest_tail  = sorted_tails[-1]

                sc = get_center(shortest_tail)
                lc = get_center(longest_tail)
                dist_tails = euclidean(sc, lc)
                dist_tails_text = f"{dist_tails:.1f}px"

                mid_tails = (int((sc[0] + lc[0]) / 2), int((sc[1] + lc[1]) / 2))

                cv2.line(annotated_frame,
                         (int(sc[0]), int(sc[1])),
                         (int(lc[0]), int(lc[1])),
                         (255, 165, 0), 2)
                cv2.putText(annotated_frame, dist_tails_text,
                            (mid_tails[0], mid_tails[1] - 8),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 165, 0), 2)

            elif len(tails) == 1:
                # Only one tail — still track it as shortest for clamp distance
                shortest_tail = tails[0]

            # Distance between shortest tail and nearest clamp
            dist_clamp_text = "N/A"
            if shortest_tail is not None and clamps:
                sc = get_center(shortest_tail)
                clamp_centers = [get_center(c) for c in clamps]
                nearest_clamp = min(clamp_centers, key=lambda cc: euclidean(sc, cc))
                dist_clamp = euclidean(sc, nearest_clamp)
                dist_clamp_text = f"{dist_clamp:.1f}px"

                mid_clamp = (int((sc[0] + nearest_clamp[0]) / 2),
                             int((sc[1] + nearest_clamp[1]) / 2))

                cv2.line(annotated_frame,
                         (int(sc[0]), int(sc[1])),
                         (int(nearest_clamp[0]), int(nearest_clamp[1])),
                         (0, 255, 255), 2)
                cv2.putText(annotated_frame, dist_clamp_text,
                            (mid_clamp[0], mid_clamp[1] - 8),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)

            # HUD
            cv2.putText(annotated_frame, f"Clamp Count: {clamp_count}",      (30, 50),  cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 0),   3)
            cv2.putText(annotated_frame, f"Tail Count: {tail_count}",        (30, 90),  cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 255),   3)
            cv2.putText(annotated_frame, f"Short-Long Tail: {dist_tails_text}", (30, 130), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 165, 0), 2)
            cv2.putText(annotated_frame, f"Short Tail-Clamp: {dist_clamp_text}", (30, 165), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 255), 2)

            out.write(annotated_frame)
    finally:
        cap.release()
        out.release()

    print("Finished!")

In [7]:
clips_dir = "/content/drive/MyDrive/clips"
clips = os.listdir(clips_dir)

for clip in clips:
    if clip.endswith(".mp4"):
        video_path = os.path.join(clips_dir, clip)
        video_name = os.path.splitext(clip)[0]  # removes .mp4 → "part_1"
        output_name = f"output_{video_name}"     # → "output_part1"

        print(f"Processing: {clip} → {output_name}.mp4")
        save_video_detection(video_path, output_name)
        print(f"Done: {output_name}.mp4\n")

print("All clips processed!")

Streaming output truncated to the last 5000 lines.

0: 384x640 1 Clamp, 7 Tails, 23.5ms
Speed: 5.1ms preprocess, 23.5ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 Clamp, 6 Tails, 23.5ms
Speed: 3.5ms preprocess, 23.5ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 Clamps, 5 Tails, 23.5ms
Speed: 3.8ms preprocess, 23.5ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 Clamps, 6 Tails, 23.5ms
Speed: 3.5ms preprocess, 23.5ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 Clamps, 6 Tails, 23.5ms
Speed: 3.5ms preprocess, 23.5ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 Clamps, 7 Tails, 23.5ms
Speed: 3.9ms preprocess, 23.5ms inference, 1.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 Clamps, 6 Tails, 23.5ms
Speed: 3.5ms preprocess, 23.5ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)

0: 38